# Memory Behavior Experiments — Session 03: What Happens Inside Functions

This notebook runs live experiments on how Python passes objects into functions and how mutability shapes the outcome. The key discovery is that arguments are passed by object reference: a function can mutate a caller's list in place, but reassigning its parameter leaves the caller untouched. Along the way you meet the famous mutable default-argument trap.

**The big picture.** Python never copies an argument at the call boundary. The parameter becomes another name for the same object. Whether the caller sees a change therefore depends on *how* the function uses the parameter — in-place mutation versus rebinding — not on any pass-by-value or pass-by-reference label.

**Why it matters.** Mutability rules explain a whole family of bugs: functions that unexpectedly modify their inputs, caches built from default arguments that persist across calls, and string loops that waste memory. They also explain the concrete benefits of immutable types.

**Roadmap.** In this notebook you will:
1. See a function mutate a caller's list, and another rebind without effect.
2. Trigger the mutable default-argument trap with a list default.
3. Fix it with the `None`-guard pattern.
4. Protect callers with defensive copying.
5. Explore immutability's three superpowers.
6. Compare string concatenation in a loop with `join`.

**Key definitions.** *Pass by object reference*: parameters share the argument object. *In-place mutation*: changing an object's contents, visible to all references. *Rebinding*: pointing a name at a new object, visible only locally. *Default argument*: a value evaluated once when the `def` statement runs.

## 1. Passing Objects into Functions

When you call a function, Python binds each parameter to the argument object without copying it. Inside the function the parameter and the caller's variable refer to the same object.

Two operations then behave very differently:

- **In-place mutation** (`.append`, `.extend`, `[i] = x`) changes the shared object, so the caller sees the change.
- **Rebinding** (`param = []`) only points the local name somewhere new. The caller's name still refers to the original object.

| Inside the function | Caller affected? |
|---|---|
| `lst.append(x)` | Yes — shared object changed |
| `lst = []` | No — only the local name moved |

This is the office-whiteboard analogy: if you erase the board, everyone sees it; if you hang a different board on your own wall, the original is untouched.

In [1]:
# 🔬 EXPERIMENT #1 — Caller ka whiteboard effect! 🖼️
def add_task(task_list):
    task_list.append("new office task")       # in-place — whiteboard edit 🧽

def reset_board(task_list):
    task_list = []                            # REASSIGN — sirf LOCAL tag bana!

office_board = ["water-plants", "check-gate"]
add_task(office_board)
print("add_task baad :", office_board, "← caller CHANGED! (in-place 🔥)")
reset_board(office_board)
print("reset baad    :", office_board, "← caller UNTOUCHED! (reassignment = local 🧊)")
# Expected OUTPUT:
# add_task baad : ['water-plants', 'check-gate', 'new office task'] ← caller CHANGED! (in-place 🔥)
# reset baad    : ['water-plants', 'check-gate', 'new office task'] ← caller UNTOUCHED! (reassignment = local 🧊)

add_task baad : ['water-plants', 'check-gate', 'new office task'] ← caller CHANGED! (in-place 🔥)
reset baad    : ['water-plants', 'check-gate', 'new office task'] ← caller UNTOUCHED! (reassignment = local 🧊)


## 2. The Mutable Default-Argument Trap

A default argument is evaluated **once**, when the `def` statement executes — not on each call. For immutable defaults like numbers or strings that is harmless. For a mutable default such as `[]` or `{}`, the same object is reused across every call.

So `def enroll(student, subjects=[])` creates one list when the function is defined, and every call that omits `subjects` appends to that same list. The first call returns `['math']`, the second `['math', 'math']`, and so on — values from unrelated calls accumulate.

The fix is the **None-guard** pattern: use `None` as the default and create a fresh list inside the body when nothing was passed.

```text
def enroll(student, subjects=None):
    if subjects is None:
        subjects = []          # fresh list every call
    ...
```

Rule: never use a mutable literal as a default. Use `None` and initialise inside.

In [2]:
# 🪤 EXPERIMENT #2 — THE Mutable Default-Arg Trap! (LEGEND ⭐)
def enroll(student, subjects=[]):            # 🪤 default list EK baar banti (def-time!)
    subjects.append("math")
    return subjects

print("call 1:", enroll("Sana"))
print("call 2:", enroll("Ravi"), "← RAVI SIRF math to chahiye... pehli-wali value shared?! 😱")
print("call 3:", enroll("Meena"))
# Expected OUTPUT:
# call 1: ['math']
# call 2: ['math', 'math'] ← RAVI SIRF math to chahiye... pehli-wali value shared?! 😱
# call 3: ['math', 'math', 'math']

call 1: ['math']
call 2: ['math', 'math'] ← RAVI SIRF math to chahiye... pehli-wali value shared?! 😱
call 3: ['math', 'math', 'math']


## 2.1 The None-Guard Fix

`enroll_fixed(student, subjects=None)` starts each call with `subjects` set to `None`, then replaces it with a brand-new empty list before appending. Because the new list is created inside the function body, it lives only for that call.

The result is the expected behaviour: each student gets `['math']`, with no sharing between calls. The guard costs almost nothing and makes the function's contract obvious to readers.

Apply this pattern to every mutable default — lists, dictionaries, and sets. Immutable defaults such as `0`, an empty string, or `None` are safe as they are.

In [3]:
# ✅ THE FIX — None-guard pattern (industry standard!)
def enroll_fixed(student, subjects=None):
    if subjects is None:
        subjects = []                         # har call FRESH list ✨
    subjects.append("math")
    return subjects

print("call 1:", enroll_fixed("Sana"))
print("call 2:", enroll_fixed("Ravi"), "← ab fresh-plate har bande ko! ✅")
print("call 3:", enroll_fixed("Meena"))
# Expected OUTPUT:
# call 1: ['math']
# call 2: ['math'] ← ab fresh-plate har bande ko! ✅
# call 3: ['math']

call 1: ['math']
call 2: ['math'] ← ab fresh-plate har bande ko! ✅
call 3: ['math']


## 3. Defensive Copying

Sometimes a function *should* modify its input, and sometimes it must protect the caller. When the caller's data must stay intact, the function can take a defensive copy at the top of its body: `scores = scores.copy()`.

After that line the function works on its own list, so appends and edits never reach the caller. This is the standard way to combine needing to mutate during processing with not surprising the caller.

Choose deliberately: mutate in place when the side effect is intended and documented, otherwise copy first. Silent in-place mutation of a caller's argument is one of the most common sources of hard-to-trace bugs.

In [4]:
# 🛡️ EXPERIMENT #3 — Defensive copy (caller-protect pattern)
def process_scores(scores):
    scores = scores.copy()                   # apni whiteboard banwalk, OG safe!
    scores.append(100)
    return scores

exam_scores = [80, 85, 90]
result = process_scores(exam_scores)
print("result     :", result)
print("exam_scores:", exam_scores, "← caller SAFE (defensive .copy() 🛡️)")
# Expected OUTPUT:
# result     : [80, 85, 90, 100]
# exam_scores: [80, 85, 90] ← caller SAFE (defensive .copy() 🛡️)

result     : [80, 85, 90, 100]
exam_scores: [80, 85, 90] ← caller SAFE (defensive .copy() 🛡️)


## 4. Immutability's Superpowers

Immutable objects — strings, numbers, tuples, frozensets — cannot be changed after creation. That constraint buys three concrete benefits.

| Power | What it gives | Example |
|---|---|---|
| Hashability | Safe dictionary keys and set members | `{('Sana',): 90}` |
| Thread safety | Can be shared freely, no one can corrupt it | Shared config strings |
| Predictability | A function cannot alter it in place | Caller peace of mind |

The experiments show this directly: `s.upper()` returns a new string while the original keeps its `id()`, proving it was never modified. Immutability is not a limitation to work around; it is what makes dictionary keys reliable and sharing safe. Mutables trade that safety for fast, flexible edits.

In [5]:
# 🦸 Immutability superpowers — STRINGS demo
s = "hero"
hero_id = id(s)

# string "badalna" = naya hi banana — but predictable, hashable, shareable-safe!
upper_s = s.upper()
print("upper        :", upper_s, "| original:", s)
print("OG id same?  :", id(s) == hero_id, "← kabhi modify hi nahi hui (stone-tablet! 🪨)")

# dict-key ho sakta — immutable superpower #2 LIVE:
scores = {("Sana",): 90}
print("tuple-keys OK:", scores)
# Expected OUTPUT:
# upper        : HERO | original: hero
# OG id same?  : True ← kabhi modify hi nahi hui (stone-tablet! 🪨)
# tuple-keys OK: {('Sana',): 90}

upper        : HERO | original: hero
OG id same?  : True ← kabhi modify hi nahi hui (stone-tablet! 🪨)
tuple-keys OK: {('Sana',): 90}


## 5. String Building: Concat Loop vs join

Strings are immutable, so `badge += p` inside a loop builds a **new** string on every iteration. For a short list the cost is invisible, but for thousands of parts the repeated allocation dominates.

The efficient idiom is `''.join(parts)`, which walks the parts once and allocates a single result string. Use it whenever you assemble text from a collection:

```text
slow : total = ''
       for p in parts:
           total += p
fast : total = ''.join(parts)
```

If you need separators or formatting, build a list of pieces and join once. As a habit, treat `join` as the default and `+=` in a loop as a smell.

In [6]:
# ⚡ EXPERIMENT #4 — concat-loop vs join (string-surgery cost!)
parts = ["chai", "samosa", "lassi"]

# WAY-A: concat-loop — har step NEW string (n strings = n objects!) 🪨🪨🪨
badge = ""
for p in parts:
    badge += p + ", "
print("concat loop  :", badge)

# WAY-B: join — EK hi final string create 🪨 (efficient!)
badge2 = ", ".join(parts)
print("join         :", badge2, "← ek-shot memory-friendly way!")
# Note: chhote sizes pe farak chupcha; LOOPS me built-up strings ke liye join = golden rule ⭐
# Expected OUTPUT:
# concat loop  : chai, samosa, lassi, 
# join         : chai, samosa, lassi ← ek-shot memory-friendly way!

concat loop  : chai, samosa, lassi, 
join         : chai, samosa, lassi ← ek-shot memory-friendly way!


## 6. Immutability's Three Superpowers

| Power | What it gives | Example |
|---|---|---|
| Hashability | Can be used as dict keys or set members | `{('Sana',): 90}` |
| Thread safety | Safe to share across threads | Configuration strings |
| Predictability | No in-place changes inside functions | Stable caller data |

Mutable types have their own advantages — fast in-place edits, easy growth — and are the right choice when data genuinely changes. The skill is choosing deliberately: immutable by default for shared or key-like data, mutable when you need to edit in place. This table is the why-bother answer to the whole topic.

## ⚠️ 5 Common Mistakes (Behavior Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | `def f(box=[])` with a mutable default | `def f(box=None)` + the None-guard 🪤✅ |
| 2 | Guessing caller-change vs local-only inside a function | In-place=caller-change 🔥; reassign=local-only 🧊; want safety? `.copy()` 🛡️ |
| 3 | `+=` in a string-building loop | `"".join(parts)` ⭐ |
| 4 | Not using the immutable advantage | dict-keys/thread-share/predictability — exploit the 3 powers! 🦸 |
| 5 | Passing an object and trusting OG-safe | Functions share references — design accordingly! |

> 🎯 **Iron rule:** keep default args immutable; if functions poke mutables in-place,
> document the call-out; build big strings with join! 🧱

## Summary

- Arguments are passed by object reference; a parameter is another name for the caller's object.
- In-place mutation reaches the caller, while rebinding stays local.
- A mutable default argument is created once and shared across calls — use `None` plus an in-body initialisation.
- Defensive copying protects a caller when a function must mutate while processing.
- Immutability provides hashability, thread safety, and predictability.
- Build strings with `join`, not with `+=` in a loop.

These rules explain most why-did-my-data-change bugs and point directly to the safe patterns that prevent them.